In [4]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
env.run(["main.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE


In [5]:
import math
from  kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def agent(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves
    
    for mine in my_planets:
        nearest = min(targets, key=lambda t: math.hypot(mine.x - t.x, mine.y - t.y))
        angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
        moves.append([mine.id, angle, 1])

    return moves

In [6]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
env.run([agent, "random"])

final = env.steps[-1]

for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

env.render(mode="html", filename="replay.html")
print("Open replay.html in your browser")

Player 0: reward=1, status=DONE
Player 1: reward=-1, status=DONE
Open replay.html in your browser


In [9]:
html = env.render(mode="html")
print(type(html))
print(len(html))

with open("replay.html", "w") as f:
    f.write(html)
print("done")

<class 'str'>
108686117
done


In [10]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet
import numpy as np

def how_many_send(my_planet: Planet,
                  enemy_planet: Planet,
                  spare: int,
                  obs):
    def planet_pos(t):
        if not orbits:
            return enemy_planet.x, enemy_planet.y
        a = t * ang_vel
        dx, dy = enemy_planet.x - 50, enemy_planet.y - 50
        px = 50 + math.cos(a) * dx - math.sin(a) * dy
        py = 50 + math.sin(a) * dx + math.cos(a) * dy
        return px, py

    def f(t):
        px, py = planet_pos(t)
        return math.hypot(px - my_planet.x, py - my_planet.y) - (my_planet.radius + 0.1) - t * speed

    l_fleet_size, r_fleet_size = enemy_planet.ships + spare, my_planet.ships
    sun_dist = math.hypot(enemy_planet.x - 50, enemy_planet.y - 50)

    ang_vel = obs.get("angular_velocity", 0) if isinstance(obs, dict) else obs.angular_velocity

    orbits = False
    if sun_dist + enemy_planet.radius < 50:
        orbits = True

    ship_max_speed = 6
    best = 1e9
    angle = 0

    while l_fleet_size <= r_fleet_size:
        ships = (l_fleet_size + r_fleet_size) // 2
        
        speed = 1.0 + (ship_max_speed - 1.0) * (math.log(ships) / math.log(1000))**1.5
        speed = min(speed, ship_max_speed)

        found_angle = 0.0
        moves = -1
        feasible = False

        prev = f(0)  # positive: planet starts away from us
        for t in range(1, 101):
            cur = f(t)
            if cur <= 0 and prev > 0:
                # root is in [t-1, t]; refine with a few bisection steps
                lo, hi = t - 1, t
                for _ in range(40):
                    mid = (lo + hi) / 2
                    if f(mid) > 0:
                        lo = mid
                    else:
                        hi = mid
                t_star = (lo + hi) / 2

                px, py = planet_pos(t_star)
                
                sx, sy = px - my_planet.x, py - my_planet.y
                seg2 = sx * sx + sy * sy
                u = ((50 - my_planet.x) * sx + (50 - my_planet.y) * sy) / seg2
                u = max(0.0, min(1.0, u))
                cxs, cys = my_planet.x + u * sx, my_planet.y + u * sy
                if math.hypot(cxs - 50, cys - 50) <= 10.0:
                    prev = cur
                    continue

                found_angle = math.atan2(py - my_planet.y, px - my_planet.x)
                moves = t_star
                feasible = True
                break
            prev = cur


        if feasible:
            prod = enemy_planet.production * moves if enemy_planet.owner != -1 else 0
            if ships >= enemy_planet.ships + prod + spare:
                best = ships
                angle = found_angle
                r_fleet_size = ships - 1
            else:
                l_fleet_size = ships + 1
        else:
            l_fleet_size = ships + 1

    return best, angle

In [15]:
from collections import namedtuple

Fleet = namedtuple("Fleet", ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

def fleet_planet_collision(obs):
    player     = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_fleets = obs.get("fleets", []) if isinstance(obs, dict) else obs.fleets
    raw_planet = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    ang_vel    = obs.get("angular_velocity", 0) if isinstance(obs, dict) else obs.angular_velocity

    fleets  = [Fleet(*f) for f in raw_fleets]
    planets = [Planet(*p) for p in raw_planet]

    my_fleets = [f for f in fleets if f.owner == player]
    if not my_fleets:
        return []

    static, orbiting = [], []
    for p in planets:
        if math.hypot(p.x - 50, p.y - 50) + p.radius < 50:
            orbiting.append(p)
        else:
            static.append((p.id, p.x, p.y, p.radius))

    MAX_T = 101

    orbit_pos = []
    for t in range(MAX_T):
        a = t * ang_vel
        ca, sa = math.cos(a), math.sin(a)
        row = []
        for p in orbiting:
            dx, dy = p.x - 50, p.y - 50
            row.append((p.id, 50 + ca*dx - sa*dy, 50 + sa*dx + ca*dy, p.radius))
        orbit_pos.append(row)

    def swept_pair_hit(ax, ay, bx, by, p0x, p0y, p1x, p1y, r):
        d0x, d0y = ax - p0x, ay - p0y
        dvx = (bx - ax) - (p1x - p0x)
        dvy = (by - ay) - (p1y - p0y)
        a = dvx*dvx + dvy*dvy
        b = 2.0*(d0x*dvx + d0y*dvy)
        c = d0x*d0x + d0y*d0y - r*r
        if a < 1e-12:
            return c <= 0.0
        disc = b*b - 4.0*a*c
        if disc < 0.0:
            return False
        sq = math.sqrt(disc)
        return (-b + sq)/(2.0*a) >= 0.0 and (-b - sq)/(2.0*a) <= 1.0

    ship_max_speed = 6
    hits = []

    for f in my_fleets:
        speed = 1.0 + (ship_max_speed-1.0)*(math.log(f.ships)/math.log(1000))**1.5
        speed = min(speed, ship_max_speed)
        dx, dy = math.cos(f.angle)*speed, math.sin(f.angle)*speed

        fx, fy = f.x, f.y
        hit = False
        for t in range(1, MAX_T):
            nfx, nfy = fx + dx, fy + dy
            seg_r = speed + 0.5 

            for pid, px, py, pr in static:
                mx, my = (fx+nfx)*0.5, (fy+nfy)*0.5
                if (px-mx)**2 + (py-my)**2 > (seg_r + pr)**2:
                    continue
                if swept_pair_hit(fx, fy, nfx, nfy, px, py, px, py, pr):
                    hits.append((f.id, pid, t)); hit = True; break
            if hit: break

            old_row, new_row = orbit_pos[t-1], orbit_pos[t]
            mx, my = (fx+nfx)*0.5, (fy+nfy)*0.5
            for (pid, p0x, p0y, pr), (_, p1x, p1y, _) in zip(old_row, new_row):
                if (p1x-mx)**2 + (p1y-my)**2 > (seg_r + pr + 3)**2:
                    continue
                if swept_pair_hit(fx, fy, nfx, nfy, p0x, p0y, p1x, p1y, pr):
                    hits.append((f.id, pid, t)); hit = True; break
            if hit: break

            fx, fy = nfx, nfy

    return hits

In [ ]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planet = obs.get("planets", 0) if isinstance(obs, dict) else obs.planets
    comets = obs.get("comet_planet_ids", 0) if isinstance(obs, dict) else obs.comet_planet_ids

    planets = [Planet(*p) for p in raw_planet]

    mine = [p for p in planets if p.owner == player]
    target = [p for p in planets if p.owner != player]


    if len(target) == 0:
        return moves
    
    for planet in mine:
        nearest = -1
        dist = 1e9
        
        for p in  target:
            is_comet = False
            for c in comets:
                if p.id == c:
                    is_comet = True

            if (p.x - planet.x)**2 + (p.y - planet.y)**2 < dist and p.ships < planet.ships and not is_comet:
                nearest = p.id
                dist = (p.x - planet.x)**2 + (p.y - planet.y)**2 
        
        for p in target:
            if p.id == nearest:
                ships, angle = how_many_send(planet, p, 1, obs)

                if ships <= planet.ships:
                    moves.append([planet.id, angle, ships])

    return moves

In [ ]:
def most_production(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planet = obs.get("planets", 0) if isinstance(obs, dict) else obs.planets
    comets = obs.get("comet_planet_ids", 0) if isinstance(obs, dict) else obs.comet_planet_ids

    planets = [Planet(*p) for p in raw_planet]

    mine = [p for p in planets if p.owner == player]
    target = [p for p in planets if p.owner != player]

    if len(target) == 0:
        return moves
    
    target = sorted(target, key=lambda x : x.production)
    hitting = fleet_planet_collision(obs)

    for p in mine:
        for i in range(len(target) - 1, -1, -1):
            ships, angle = how_many_send(p, target[i], 1, obs)

            skip_this_planet = False

            for c in comets:
                if target[i].id == c:
                    skip_this_planet = True

            for ship in hitting:
                if ship[1] == target[i].id:
                    skip_this_planet = True
            
            if skip_this_planet:
                continue

            if ships < p.ships:
                moves.append([p.id, angle, max(ships, p.ships // 2)])
                break

    return moves

In [30]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

def nearest_planet_smart(obs):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planet = obs.get("planets", 0) if isinstance(obs, dict) else obs.planets
    comets = obs.get("comet_planet_ids", 0) if isinstance(obs, dict) else obs.comet_planet_ids

    planets = [Planet(*p) for p in raw_planet]

    mine = [p for p in planets if p.owner == player]
    target = [p for p in planets if p.owner != player]


    if len(target) == 0:
        return moves
    
    hitting = fleet_planet_collision(obs)
    
    for planet in mine:
        nearest = -1
        dist = 1e9
        
        for p in  target:       
            is_bad = False
            for c in comets:
                if p.id == c:
                    is_bad = True

            for ship in hitting:
                if ship[1] == p.id:
                    is_bad = True

            if (p.x - planet.x)**2 + (p.y - planet.y)**2 < dist and p.ships < planet.ships and not is_bad:
                nearest = p.id
                dist = (p.x - planet.x)**2 + (p.y - planet.y)**2 
        
        for p in target:
            if p.id == nearest:
                ships, angle = how_many_send(planet, p, 1, obs)

                if ships <= planet.ships:
                    moves.append([planet.id, angle, ships])

    return moves

In [31]:
from kaggle_environments import make

env = make("orbit_wars", debug=True)
env.run([nearest_planet, nearest_planet_smart])

final = env.steps[-1]

for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")

html = env.render(mode="html")
print(type(html))
print(len(html))

with open("replay.html", "w") as f:
    f.write(html)
print("done")

Player 0: reward=-1, status=DONE
Player 1: reward=1, status=DONE
<class 'str'>
6422333
done
